<a href="https://colab.research.google.com/github/dennisddschulz/cas-artificial-intelligence/blob/main/08_drl_einstieg/02_DRL_Policy_Improvement_taxi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DRL Intro: MDP, Return, V/Q/Advantage – Monte-Carlo Evaluation (FrozenLake)

Ziel:
1) Episode sammeln (Rollout)
2) Returns G_t berechnen
3) Monte-Carlo Schätzung von V(s) und Q(s,a)
4) Advantage A(s,a) berechnen
5) Aus Q eine ε-greedy Policy ableiten
6) Zeigen, dass sich V(start) verbessert (Policy Improvement)


In [26]:
!pip -q install gymnasium

import gymnasium as gym
import numpy as np
from collections import defaultdict

SEED = 42
rng = np.random.default_rng(SEED)


In [27]:
import gymnasium as gym

env = gym.make("Taxi-v3")

s0, info = env.reset(seed=SEED)
nS = env.observation_space.n
nA = env.action_space.n

print("nS:", nS, "nA:", nA, "start:", s0)

nS: 500 nA: 6 start: 386


## Policy und Rollout

- Policy: Funktion, die aus Zustand s eine Aktion a wählt.
- Rollout: wir lassen Agent+Env laufen und speichern (s, a, r) pro Schritt.


In [28]:
def random_policy(s, nA):
    return int(rng.integers(nA))

def rollout_episode(env, policy_fn, max_steps=200, seed=0):
    traj = []  # list of (s, a, r)
    s, _ = env.reset(seed=seed)
    for _ in range(max_steps):
        a = policy_fn(s, env.action_space.n)
        s2, r, terminated, truncated, _ = env.step(a)
        traj.append((s, a, r))
        s = s2
        if terminated or truncated:
            break
    return traj

traj = rollout_episode(env, random_policy, seed=SEED)
print("episode length:", len(traj))
print("first steps:", traj[:8])


episode length: 200
first steps: [(386, 0, -1), (486, 4, -10), (486, 3, -1), (466, 2, -1), (486, 2, -1), (486, 5, -10), (486, 0, -1), (486, 4, -10)]


## Returns berechnen

Return G_t ist die discounted Summe der zukünftigen Rewards ab Schritt t.
Wir berechnen das rückwärts:
G = 0
G <- r + gamma * G


In [29]:
def compute_returns(traj, gamma=0.99):
    G = 0.0
    returns = []
    # HA - Warum reversed?
    for (s, a, r) in reversed(traj):
        G = r + gamma * G
        returns.append(G)
    returns.reverse()
    return returns

gamma = 0.99
Gs = compute_returns(traj, gamma=gamma)
list(zip(traj[:8], Gs[:8]))


[((386, 0, -1), -378.5932359691751),
 ((486, 4, -10), -381.40730905977284),
 ((486, 3, -1), -375.1588980401746),
 ((466, 2, -1), -377.93828084866124),
 ((486, 2, -1), -380.74573823097097),
 ((486, 5, -10), -383.58155376865756),
 ((486, 0, -1), -377.3551048168258),
 ((486, 4, -10), -380.1566715321473)]

## MC Evaluation von V(s)

First-Visit MC:
- pro Episode zählt nur das erste Auftreten eines Zustands s
- V(s) = Durchschnitt der beobachteten Returns in s

```
seen = set()
for t, (s, a, r) in enumerate(traj):
    if s in seen:
        continue
    seen.add(s)
    V[s] += G[t]
```

Every-Visit MC:
- pro Episode zählt jedes Auftreten eines Zustands s
- V(s) ist der Durchschnitt der beobachteten Return über alle Besuche von s in allen Episoden.
```
for t, (s, a, r) in enumerate(traj):
    V[s] += G[t]
```


In [30]:
def mc_evaluate_V(env, policy_fn, episodes=3000, gamma=0.99, seed=0):
    returns_sum = defaultdict(float)
    returns_count = defaultdict(int)

    for ep in range(episodes):
        traj = rollout_episode(env, policy_fn, seed=seed + ep)
        Gs = compute_returns(traj, gamma=gamma)

        seen = set()
        for t, (s, a, r) in enumerate(traj):
            if s in seen:
                continue
            seen.add(s)
            returns_sum[s] += Gs[t]
            returns_count[s] += 1

    V = {s: returns_sum[s] / returns_count[s] for s in returns_count}
    return V

V_rand = mc_evaluate_V(env, random_policy, episodes=4000, gamma=gamma, seed=SEED)

s0, _ = env.reset(seed=SEED)
print("V_random(start):", round(V_rand.get(s0, 0.0), 4))


V_random(start): -274.5876


In [31]:
V_rand

{386: -274.5875854975096,
 286: -274.6570730753294,
 186: -261.43646617310213,
 86: -244.6839570109046,
 98: -220.91875896740652,
 198: -210.1388520314953,
 298: -213.38572535295398,
 398: -219.08419066823433,
 178: -211.8103835067926,
 158: -213.82538459138465,
 58: -202.57340807558228,
 78: -215.72266603932084,
 258: -200.50040652762888,
 238: -186.93000906022687,
 218: -154.85102042012883,
 318: -96.56067950151974,
 418: -61.48429852168677,
 118: -183.04477982216838,
 18: -194.08305227493776,
 2: -221.84981762021062,
 22: -235.02244026874897,
 102: -246.18633715560497,
 202: -260.8118718280788,
 222: -277.2263177186641,
 242: -296.16599766954795,
 262: -299.64182308689584,
 362: -282.07769696430665,
 462: -277.7506215260124,
 382: -286.40379214915333,
 482: -273.4053691625837,
 282: -294.9706581595195,
 162: -293.3702343337955,
 142: -287.2775886449516,
 122: -244.50745547043937,
 322: -275.50317729429895,
 342: -277.488756074186,
 442: -266.7049809179074,
 422: -265.47740216424353,

## MC Evaluation von Q(s,a)

Analog:
- wir mitteln Returns pro (s,a)
- daraus können wir greedy / ε-greedy Policies bauen


In [32]:
def mc_evaluate_Q(env, policy_fn, episodes=6000, gamma=0.99, seed=0):
    returns_sum = defaultdict(float)
    returns_count = defaultdict(int)

    for ep in range(episodes):
        traj = rollout_episode(env, policy_fn, seed=seed + ep)
        Gs = compute_returns(traj, gamma=gamma)

        #seen_sa = set()
        for t, (s, a, r) in enumerate(traj):
            key = (s, a)
            #if key in seen_sa:
            #    continue
            #seen_sa.add(key)
            returns_sum[key] += Gs[t]
            returns_count[key] += 1

    Q = {k: returns_sum[k] / returns_count[k] for k in returns_count}
    return Q

Q_rand = mc_evaluate_Q(env, random_policy, episodes=8000, gamma=gamma, seed=SEED)
print("Q entries:", list(Q_rand.items())[:5])


Q entries: [((386, 3), -224.91790701563625), ((366, 4), -229.76888378345163), ((366, 2), -221.270364428927), ((366, 5), -237.90524895738199), ((366, 1), -232.36598637610336)]


## Advantage A(s,a)

A(s,a) = Q(s,a) - V(s)
Interpretation: wie viel besser/schlechter ist Aktion a gegenüber dem "Durchschnitt" in s.


In [33]:
def advantage(V, Q, s, a):
    return Q.get((s, a), 0.0) - V.get(s, 0.0)

# Advantage im Startzustand für alle Aktionen
adv_start = [(a, advantage(V_rand, Q_rand, s0, a)) for a in range(nA)]
adv_start


[(0, 49.53045242989555),
 (1, 47.90052105606969),
 (2, 39.800194141735574),
 (3, 49.669678481873376),
 (4, 32.80072900524215),
 (5, 39.715634004970724)]

## Policy Improvement: ε-greedy aus Q

- greedy: a = argmax_a Q(s,a)
- ε-greedy: mit Wahrscheinlichkeit ε zufällig, sonst greedy

Dann evaluieren wir die neue Policy wieder mit MC und vergleichen V(start).


In [34]:
def epsilon_greedy_policy_from_Q(Q, nA, eps=0.1):
    def policy(s, nA_ignored=None):
        if rng.random() < eps:
            return int(rng.integers(nA))
        qs = [Q.get((s, a), 0.0) for a in range(nA)]
        return int(np.argmax(qs))
    return policy

pi_eps = epsilon_greedy_policy_from_Q(Q_rand, nA, eps=0.1)

V_eps = mc_evaluate_V(env, pi_eps, episodes=4000, gamma=gamma, seed=SEED)
print("V_random(start):", round(V_rand.get(s0, 0.0), 4))
print("V_eps(start):   ", round(V_eps.get(s0, 0.0), 4))


V_random(start): -274.5876
V_eps(start):    -108.2512


## Mini Loop: wiederholte Verbesserung (Iteration)

Wir wiederholen:
1) Q unter aktueller Policy schätzen
2) neue ε-greedy Policy bauen
3) V(start) loggen

Achtung: Das ist noch nicht "Policy Iteration" im strengen Sinn,
aber zeigt sehr gut die Grundidee: Bessere Wertschätzungen (V/Q) → bessere Entscheidungsgrundlage → verbesserte Policy


In [35]:
def policy_improvement_loop(env, init_policy, iters=5, eps=0.1, episodes_Q=6000, episodes_V=3000, gamma=0.99, seed=0):
    policy = init_policy
    history = []

    s0, _ = env.reset(seed=seed)

    for k in range(iters):
        Q = mc_evaluate_Q(env, policy, episodes=episodes_Q, gamma=gamma, seed=seed + 1000*k)
        policy = epsilon_greedy_policy_from_Q(Q, env.action_space.n, eps=eps)
        V = mc_evaluate_V(env, policy, episodes=episodes_V, gamma=gamma, seed=seed + 2000*k)
        history.append((k, V.get(s0, 0.0)))

    return history

hist = policy_improvement_loop(env, random_policy, iters=6, eps=0.1, episodes_Q=6000, episodes_V=3000, gamma=gamma, seed=SEED)
hist


[(0, -91.58010374181248),
 (1, -89.52141209060765),
 (2, -738.4565059505408),
 (3, -167.15676729468933),
 (4, -95.38374542653546),
 (5, -203.09616865918747)]

## Was haben wir heute gelernt?

- Reward vs Return: Return ist das Ziel, nicht der einzelne Reward.
- V(s) und Q(s,a) sind Erwartungswerte von Returns.
- Monte-Carlo schätzt diese Werte aus Episoden (ohne Modell von P).
- Advantage erklärt "wie gut ist diese Aktion relativ zum Durchschnitt in s".
- Aus Q kann man eine bessere Policy ableiten (ε-greedy).


## Policy Improvement - Wie lernt der Agent?

In [36]:
def epsilon_linear(k, eps0=0.3, eps_min=0.02, decay_steps=10):
    # fällt linear ab von eps0 zu eps_min über decay_steps Iterationen
    eps = eps0 - (eps0 - eps_min) * (k / decay_steps)
    return float(max(eps_min, eps))

def epsilon_exp(k, eps0=0.3, eps_min=0.02, alpha=0.85):
    eps = eps0 * (alpha ** k)
    return float(max(eps_min, eps))


In [37]:
def greedy_action_from_Q(Q, s, nA):
    qs = [Q.get((s, a), 0.0) for a in range(nA)]
    return int(np.argmax(qs))

def make_epsilon_greedy_policy(Q, nA, eps):
    def policy(s, nA_ignored=None):
        if rng.random() < eps:
            return int(rng.integers(nA))
        return greedy_action_from_Q(Q, s, nA)
    return policy


In [38]:
def evaluate_success_rate(env, policy_fn, episodes=1000, seed=0, max_steps=200):
    successes = 0
    for ep in range(episodes):
        s, _ = env.reset(seed=seed + ep)
        total_reward = 0
        for _ in range(max_steps):
            a = policy_fn(s, env.action_space.n)
            s, r, terminated, truncated, _ = env.step(a)
            total_reward += r
            if terminated or truncated:
                break
        if total_reward > 0:
            successes += 1
    return successes / episodes


In [39]:
def policy_improvement_loop_mc_control_decay(
    env,
    init_policy,
    iters=8,
    eps_schedule="linear",      # "linear" oder "exp"
    eps0=0.3,
    eps_min=0.02,
    decay_steps=10,             # für linear
    alpha=0.85,                 # für exp
    episodes_Q=6000,
    episodes_V=3000,
    eval_episodes=1000,
    gamma=0.99,
    seed=0
):
    policy = init_policy
    history = []
    s0, _ = env.reset(seed=seed)

    for k in range(iters):
        # 1) Evaluate current policy -> estimate Q
        Q = mc_evaluate_Q(env, policy, episodes=episodes_Q, gamma=gamma, seed=seed + 1000*k)

        # 2) Choose epsilon for this iteration
        if eps_schedule == "linear":
            eps = epsilon_linear(k, eps0=eps0, eps_min=eps_min, decay_steps=decay_steps)
        elif eps_schedule == "exp":
            eps = epsilon_exp(k, eps0=eps0, eps_min=eps_min, alpha=alpha)
        else:
            raise ValueError("eps_schedule must be 'linear' or 'exp'")

        # 3) Improve policy using greedy(max) with epsilon exploration
        policy = make_epsilon_greedy_policy(Q, env.action_space.n, eps=eps)

        # 4) Track progress
        V = mc_evaluate_V(env, policy, episodes=episodes_V, gamma=gamma, seed=seed + 2000*k)
        v0 = float(V.get(s0, 0.0))
        sr = evaluate_success_rate(env, policy, episodes=eval_episodes, seed=seed + 3000*k)

        history.append({"iter": k, "eps": eps, "V(start)": v0, "success_rate": sr})
        print(f"iter={k:02d}  eps={eps:.3f}  V(start)={v0:.4f}  success_rate={sr:.3f}")

    return policy, Q, history


In [40]:
trained_epsilon_greedy_policy_func, q_final, hist = policy_improvement_loop_mc_control_decay(
env, random_policy, iters=50, eps_schedule="linear", eps0=0.9, eps_min=0.01, decay_steps=20,
episodes_Q=100000, episodes_V=5000, eval_episodes=1000, gamma=gamma, seed=SEED
)

iter=00  eps=0.500  V(start)=-128.1925  success_rate=0.016
iter=01  eps=0.478  V(start)=-75.6512  success_rate=0.035
iter=02  eps=0.456  V(start)=-31.5028  success_rate=0.046
iter=03  eps=0.433  V(start)=-32.1614  success_rate=0.076
iter=04  eps=0.411  V(start)=-23.9224  success_rate=0.089
iter=05  eps=0.389  V(start)=-27.4951  success_rate=0.118
iter=06  eps=0.367  V(start)=-26.4948  success_rate=0.108
iter=07  eps=0.344  V(start)=-22.8165  success_rate=0.116
iter=08  eps=0.322  V(start)=-21.2161  success_rate=0.114
iter=09  eps=0.300  V(start)=-71.9662  success_rate=0.166
iter=10  eps=0.278  V(start)=-20.9189  success_rate=0.160
iter=11  eps=0.256  V(start)=-129.4597  success_rate=0.154
iter=12  eps=0.233  V(start)=-45.6526  success_rate=0.196
iter=13  eps=0.211  V(start)=-100.2291  success_rate=0.201
iter=14  eps=0.189  V(start)=-57.8353  success_rate=0.181
iter=15  eps=0.167  V(start)=-104.0510  success_rate=0.200
iter=16  eps=0.144  V(start)=-70.3252  success_rate=0.198
iter=17  e

In [41]:
q_final

{(361, 0): -79.47572917273676,
 (461, 0): -73.60053571617166,
 (461, 2): -73.4049768521634,
 (481, 3): -76.04527759956467,
 (461, 5): -82.79835162136382,
 (461, 1): -73.76847834955417,
 (461, 4): -84.93914766074573,
 (461, 3): -72.59586229670047,
 (481, 4): -93.65019042359724,
 (251, 3): -98.18852482314507,
 (231, 1): -97.91908674257827,
 (131, 4): -94.51758520027947,
 (131, 3): -91.20225305548874,
 (111, 3): -84.79355159748619,
 (111, 2): -84.13775987359878,
 (111, 1): -85.73354938255729,
 (11, 0): -86.06717037704286,
 (111, 0): -79.96420192110402,
 (211, 0): -72.87904690397127,
 (311, 3): -65.42568136906196,
 (311, 5): -74.83993525340449,
 (311, 2): -65.7056298526975,
 (311, 0): -57.03795746143669,
 (411, 0): -53.76761965160277,
 (411, 1): -60.93118705817988,
 (411, 4): -36.95336439494677,
 (419, 1): -36.06069070569906,
 (319, 1): -33.39570014134925,
 (219, 3): -31.384163375832784,
 (208, 0): 9.127832912450383,
 (308, 0): 10.712718975895859,
 (408, 4): 12.050219047472186,
 (416, 1): 

In [42]:

greedy_trained_policy = make_epsilon_greedy_policy(q_final, env.action_space.n, eps=0.0)

In [43]:
_, q_evolution, hist = policy_improvement_loop_mc_control_decay(env, greedy_trained_policy, seed=SEED)

iter=00  eps=0.300  V(start)=-156.7903  success_rate=0.038
iter=01  eps=0.272  V(start)=-140.5305  success_rate=0.002
iter=02  eps=0.244  V(start)=-129.6435  success_rate=0.009
iter=03  eps=0.216  V(start)=-188.9058  success_rate=0.007
iter=04  eps=0.188  V(start)=-125.7721  success_rate=0.002
iter=05  eps=0.160  V(start)=-356.1035  success_rate=0.015
iter=06  eps=0.132  V(start)=-245.0250  success_rate=0.004
iter=07  eps=0.104  V(start)=-118.9561  success_rate=0.006


## Run the optimized policy improvement loop

### Subtask:
Execute the `policy_improvement_loop_mc_control_decay` function with updated parameters for more episodes and iterations to improve learning.


# Task
Modify the `policy_improvement_loop_mc_control_decay` call in cell `iwIgZfxKFy0n` to set `gamma=0.999`, `eps_min=0.005`, and `decay_steps=45`, keeping `iters=50`, `eps_schedule="linear"`, `eps0=0.7`, `episodes_Q=100000`, `episodes_V=5000`, `eval_episodes=1000`, and `seed=SEED`. After executing the cell, review the final `V(start)` and `success_rate` values to determine if the target of a success rate above 50% has been met; if not, consider exploring Q-Learning.

**Reasoning**:
The subtask requires modifying an existing function call in a specific cell. I will update the `policy_improvement_loop_mc_control_decay` function call in cell `iwIgZfxKFy0n` with the new parameter values for `gamma`, `eps_min`, and `decay_steps` while keeping other parameters as specified.

